# 05 — Case Study: JAK Kinase Family

**Deep dive into model performance on the Janus kinase (JAK) subfamily.**

The JAK family (JAK1, JAK2, JAK3, TYK2) was selected because it offers:
- **36,059 records** across 4 well-characterized members
- **2,498 compounds** tested against all 4 JAKs (enabling selectivity analysis)
- **Clinical relevance**: FDA-approved drugs (tofacitinib, baricitinib, ruxolitinib)
- **Rich compound overlap**: 81% of JAK1 compounds also tested on JAK2

### Questions addressed:
1. Which JAK member is hardest to predict, and why?
2. Do activity cliffs (similar structures, different activity) drive prediction errors?
3. Can protein-aware models predict JAK selectivity better than fingerprint-based models?
4. How does JAK-specific performance compare to dataset-wide metrics?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image

FIGURES = Path("../results/figures")
TABLES = Path("../results/tables")

# Load pre-computed Phase 6 results
jak_metrics = pd.read_csv(TABLES / "phase6_jak_per_target.csv")
jak_selectivity = pd.read_csv(TABLES / "phase6_jak_selectivity.csv")
jak_cliffs = pd.read_csv(TABLES / "phase6_jak_activity_cliffs.csv")
jak_worst = pd.read_csv(TABLES / "phase6_jak_worst_predictions.csv")

print(f"Per-target metrics: {len(jak_metrics)} rows")
print(f"Selectivity results: {len(jak_selectivity)} rows")
print(f"Activity cliff pairs: {len(jak_cliffs)}")
print(f"Worst predictions: {len(jak_worst)} rows")

## JAK Subfamily Overview

The JAK family shares a conserved kinase domain but differs in substrate specificity and tissue distribution. JAK1 and JAK2 are ubiquitously expressed and mediate signaling for many cytokines, while JAK3 is restricted to hematopoietic cells and TYK2 is involved in interferon signaling. Selective JAK inhibition is a major drug design goal — pan-JAK inhibitors (tofacitinib) cause more side effects than selective ones (upadacitinib for JAK1).

In [ ]:
# Activity distributions and compound overlap
display(Image(filename=str(FIGURES / "phase6_jak_violins.png"), width=700))
display(Image(filename=str(FIGURES / "phase6_jak_compound_overlap.png"), width=500))

## Model Performance on JAK Family

**Key finding**: ESM-FP MLP dominates on random splits (gold-bordered cells), but the advantage is specific to the random split where target identity leaks. JAK3 is consistently the hardest target across all models.

In [ ]:
# Per-target RMSE heatmaps (random and scaffold splits)
display(Image(filename=str(FIGURES / "phase6_jak_heatmap_random.png"), width=700))
display(Image(filename=str(FIGURES / "phase6_jak_heatmap_scaffold.png"), width=700))

# Numeric comparison table
random_metrics = jak_metrics[jak_metrics["split"] == "random"][
    ["model", "gene_symbol", "rmse", "r2", "pearson_r", "n_compounds"]
].pivot(index="model", columns="gene_symbol", values="rmse")
print("RMSE by model x JAK target (Random Split):")
display(random_metrics.round(3))

# JAK vs dataset-wide
display(Image(filename=str(FIGURES / "phase6_jak_vs_global.png"), width=800))

## Activity Cliffs and Compound-Level Analysis

**Activity cliffs** are pairs of structurally similar compounds (Tanimoto >= 0.85) with large activity differences (|delta pActivity| >= 1.5 log units). These represent the hardest cases for fingerprint-based models because the structural difference falls below the fingerprint resolution.

We found **7 activity cliff pairs** in the JAK test set. The most dramatic: two identical-fingerprint compounds with a 2.0 log-unit JAK1 activity difference.

In [ ]:
# Activity cliff scatter plots per JAK member
display(Image(filename=str(FIGURES / "phase6_jak_activity_cliffs.png"), width=900))

# Top cliff pairs
print("Top activity cliff pairs:")
display(jak_cliffs[["gene_symbol", "pactivity_1", "pactivity_2", 
                     "delta_pactivity", "tanimoto"]].head(10))

# Predicted vs actual scatter (RF vs ESM-FP MLP)
display(Image(filename=str(FIGURES / "phase6_jak_scatter_random.png"), width=900))

# Worst predictions breakdown
print("\nWorst prediction failure modes (top-20 per model, random split):")
failure_summary = jak_worst.groupby("model").agg(
    mean_error=("abs_error", "mean"),
    max_error=("abs_error", "max"),
    n_noisy=("is_noisy", "sum"),
    top_gene=("gene_symbol", lambda x: x.value_counts().index[0]),
    top_type=("standard_type", lambda x: x.value_counts().index[0]),
).round(3)
display(failure_summary)

# Target difficulty radar
display(Image(filename=str(FIGURES / "phase6_jak_radar.png"), width=600))

## Selectivity Prediction — The Key Finding

This is the most important analysis in the case study. When the same compound is tested against multiple JAK members, can the model correctly predict **which JAK** it inhibits most potently?

Fingerprint-based models see identical input for the same compound regardless of which JAK is being predicted — they can only learn the *average* activity across targets. Protein-aware models (ESM-FP MLP, Fusion) see both the ligand and the target embedding, enabling them to distinguish JAK members.

**Result**: ESM-FP MLP and Fusion achieve ~79% top-1 accuracy on the scaffold split, vs ~52% for all fingerprint-based models. This is the clearest evidence that protein embeddings add genuine value — not for absolute affinity, but for **relative selectivity across related targets**.

In [ ]:
# Selectivity prediction results
display(Image(filename=str(FIGURES / "phase6_jak_selectivity_random.png"), width=800))
display(Image(filename=str(FIGURES / "phase6_jak_selectivity_scaffold.png"), width=800))

# Numeric table
print("Selectivity prediction accuracy:")
display(jak_selectivity[["model", "split", "n_compounds", "top1_accuracy", 
                          "mean_rank_corr"]].round(3))

## Summary

### Key findings from the JAK case study:

1. **JAK targets are easier than average** — RMSE 0.60-0.94 vs dataset-wide 0.78-0.83, reflecting their large training sets and well-explored SAR

2. **JAK3 is hardest** — highest activity range, most noisy data, fewest compounds. Average RMSE across models is 0.933 vs 0.825 for TYK2

3. **Activity cliffs are rare but impactful** — only 7 pairs at Tanimoto ≥ 0.85 and |ΔpActivity| ≥ 1.5, but they represent the fundamental limit of fingerprint-based prediction

4. **Protein-aware models excel at selectivity** — ESM-FP MLP and Fusion achieve 79% top-1 accuracy for predicting which JAK a compound preferentially inhibits, vs 52% for fingerprint-based models. This is the strongest case for protein embeddings in the entire project

5. **The selectivity advantage holds under scaffold splitting** — it's not an artifact of data leakage

6. **Worst predictions are NOT driven by noisy data** — 0/20 worst predictions per model are flagged as noisy, suggesting model limitations rather than data quality issues

### Implications for JAK inhibitor design:
- Use **ESM-FP MLP or Fusion** when selectivity profiling is the goal
- Use **Random Forest** when absolute potency prediction with calibrated uncertainty is needed
- **JAK3-selective inhibitor design** is the hardest prediction task — consider augmenting with structural data